<a href="https://colab.research.google.com/github/hidrofluorik/Dataset-Google-Play-Store-Apps/blob/main/P13_Anggota_3_Varian_Davies_Bouldin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bagian 4 — Varian 3 — Penentuan k dengan Davies-Bouldin Index

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 3 |
| Dikerjakan oleh | **Anggota 3** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 saja |
| Luaran wajib | `data/hasil_varian/varian3.json`, `gambar/varian3_davies_bouldin.png` |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 4

> **Ketentuan Varian 3:** *Davies-Bouldin Index untuk tiap k, dengan Calinski-Harabasz sebagai pemeriksa silang. Tentukan k, lalu jelaskan apa yang diukur kedua indeks itu dan mengapa arah bacanya berlawanan.*

Bagian ini dikerjakan **Anggota 3 saja**, dan dinilai **perorangan**.

Varian Anda memakai **himpunan fitur dan penskalaan yang persis sama** dengan Varian 1 dan Varian 2. Yang berbeda hanya kriteria untuk memilih k. Karena itu angka ketiga varian benar-benar sebanding, dan tabel di Bagian 5 adalah perbandingan tiga arah yang sah.

Anda **tidak perlu menunggu** anggota lain. Varian ini hanya membutuhkan `data_bersih.csv`.

**Urutan kerja**

1. Jalankan 4.0 (persiapan).
2. Jalankan 4.1. Anda hanya membutuhkan `data_bersih.csv`.
3. Jalankan 4.2.
4. **Tulis sendiri** ketiga jawaban di 4.3.
5. Jalankan 4.4 dan commit dengan akun Anda sendiri.

**Yang perlu dipahami lebih dulu**

**Davies-Bouldin Index** menghitung, untuk tiap klaster, rasio antara sebaran di dalam klaster itu ditambah sebaran tetangga terdekatnya, dibagi jarak antarpusat keduanya. Lalu diambil rata-ratanya. **Makin kecil makin baik**: nilai kecil berarti klaster rapat di dalam dan berjauhan satu sama lain.

**Calinski-Harabasz Index** memakai gagasan yang sama tetapi dari arah berlawanan: rasio keragaman antarklaster terhadap keragaman di dalam klaster, disesuaikan dengan jumlah klaster. **Makin besar makin baik.**

Keduanya dipakai bersama sebagai pemeriksa silang. Kalau keduanya menunjuk k yang sama, pilihan itu lebih kuat. Kalau berbeda, selisihnya harus dijelaskan.

Patokan tafsiran Davies-Bouldin yang dipakai mata kuliah ini:

| Nilai | Tafsiran |
|---|---|
| di bawah 0,8 | klaster rapat di dalam dan berjauhan antarklaster |
| 0,8 sampai 1,5 | pemisahan sedang, masih dapat dipertanggungjawabkan |
| di atas 1,5 | sebaran dalam klaster hampir sebesar jarak antarklaster |

**Satu hal yang sering disalahpahami.** Davies-Bouldin dan Silhouette tidak harus sepakat. Silhouette dihitung per baris lalu dirata-rata, sehingga sensitif terhadap baris yang berada di perbatasan. Davies-Bouldin dihitung dari pusat dan sebaran tiap klaster, sehingga lebih peka terhadap klaster yang ukurannya timpang. Perbedaan hasil justru bahan diskusi yang bagus untuk laporan.

---

## 4.0 Persiapan — **jalankan setiap kali notebook dibuka**

Sesi Colab menghapus seluruh berkasnya setiap kali runtime dimulai ulang. Karena itu hasil kerja tidak disimpan di sesi Colab, melainkan di **satu folder Google Drive** yang dipakai bersama sepanjang Pertemuan 12 sampai 15. Empat sel di bawah menghubungkan folder itu, memasang pustaka, menulis `alat.py`, dan menyiapkan fungsi pembantu. Keempatnya dijalankan lebih dulu, apa pun yang Anda kerjakan di notebook ini.

`alat.py` berisi fungsi yang dipakai ketiga varian. **Jangan mengubahnya sendiri-sendiri:** kalau satu anggota mengubah isi berkas ini, ketiga varian berhenti sebanding dan tabel perbandingan menjadi tidak sah.

In [ ]:
# Folder Drive ini adalah tempat serah-terima antar pertemuan: apa pun yang
# ditulis di sini terbaca kembali oleh notebook pertemuan berikutnya.
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/ProyekML_KlasteringKMeans'
os.makedirs(FOLDER, exist_ok=True)
os.chdir(FOLDER)      # sejak baris ini, 'data/...' berarti di dalam FOLDER

if FOLDER not in sys.path:
    sys.path.insert(0, FOLDER)   # agar 'import alat' menemukan berkas di folder ini

for d in ('data', 'data/hasil_varian', 'model', 'gambar'):
    os.makedirs(d, exist_ok=True)

print('Folder proyek:', FOLDER)
print('Isi folder   :', sorted(os.listdir()) or '(masih kosong)')

Mounted at /content/drive
Folder proyek: /content/drive/MyDrive/ProyekML_KlasteringKMeans
Isi folder   : ['__pycache__', 'alat.py', 'data', 'gambar', 'model']


In [6]:
import ast, json, os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

warnings.filterwarnings('ignore')
pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 40)

SEED = 42
ID_COL = 'CUST_ID'        # ganti sesuai kolom identitas dataset kelompok Anda
IQR_PENGALI = 1.5

print('Siap. scikit-learn', sklearn.__version__)
print('Catat versi ini. Angka yang sama nanti ditulis di requirements.txt.')

Siap. scikit-learn 1.6.1
Catat versi ini. Angka yang sama nanti ditulis di requirements.txt.


In [ ]:
%%writefile alat.py
"""alat.py -- fungsi bersama ketiga varian. Jangan diubah sendiri-sendiri."""
from __future__ import annotations

import json
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (calinski_harabasz_score,
                             davies_bouldin_score, silhouette_score)
from sklearn.preprocessing import StandardScaler

SEED = 42
RENTANG_K = range(2, 11)
BERKAS_BERSIH = 'data/data_bersih.csv'
FOLDER_HASIL = 'data/hasil_varian'

# Label dan satuan untuk antarmuka aplikasi (ketentuan Tahap 3: setiap
# masukan harus punya label dan satuan yang dipahami orang luar).
LABEL = {
    'BALANCE': ('Saldo rata-rata', 'USD'),
    'BALANCE_FREQUENCY': ('Keseringan saldo diperbarui', 'skala 0-1'),
    'PURCHASES': ('Total pembelian', 'USD'),
    'ONEOFF_PURCHASES': ('Pembelian sekali bayar', 'USD'),
    'INSTALLMENTS_PURCHASES': ('Pembelian cicilan', 'USD'),
    'CASH_ADVANCE': ('Penarikan tunai', 'USD'),
    'PURCHASES_FREQUENCY': ('Keseringan berbelanja', 'skala 0-1'),
    'ONEOFF_PURCHASES_FREQUENCY': ('Keseringan belanja sekali bayar', 'skala 0-1'),
    'PURCHASES_INSTALLMENTS_FREQUENCY': ('Keseringan belanja cicilan', 'skala 0-1'),
    'CASH_ADVANCE_FREQUENCY': ('Keseringan menarik tunai', 'skala 0-1'),
    'CASH_ADVANCE_TRX': ('Jumlah transaksi tarik tunai', 'kali'),
    'PURCHASES_TRX': ('Jumlah transaksi belanja', 'kali'),
    'CREDIT_LIMIT': ('Limit kartu', 'USD'),
    'PAYMENTS': ('Total pembayaran', 'USD'),
    'MINIMUM_PAYMENTS': ('Pembayaran minimum', 'USD'),
    'PRC_FULL_PAYMENT': ('Proporsi pelunasan penuh', 'skala 0-1'),
    'TENURE': ('Lama berlangganan', 'bulan'),
}


def label_satuan(kolom: str) -> tuple[str, str]:
    return LABEL.get(kolom, (kolom.replace('_', ' ').title(), ''))


def muat_bersih(fitur: list[str] | None = None) -> pd.DataFrame:
    """Membaca data bersih kelompok. Kolom identitas selalu dibuang."""
    df = pd.read_csv(BERKAS_BERSIH)
    # Hanya kolom numerik yang menjadi fitur pembentuk klaster.
    # Kolom identitas seperti CUST_ID tidak membawa informasi kemiripan.
    df = df.select_dtypes(include='number')
    return df[fitur] if fitur else df


def skalakan(df: pd.DataFrame):
    sc = StandardScaler()
    return sc.fit_transform(df), sc


def jalankan_k(X_scaled, rentang=RENTANG_K) -> pd.DataFrame:
    """Menjalankan K-Means untuk tiap k, mengembalikan tabel metrik."""
    baris = []
    for k in rentang:
        km = KMeans(n_clusters=k, init='k-means++', n_init=20,
                    random_state=SEED)
        lab = km.fit_predict(X_scaled)
        baris.append({
            'k': k,
            'inertia': float(km.inertia_),
            'silhouette': float(silhouette_score(X_scaled, lab)),
            'davies_bouldin': float(davies_bouldin_score(X_scaled, lab)),
            'calinski_harabasz': float(calinski_harabasz_score(X_scaled, lab)),
        })
    return pd.DataFrame(baris)


def titik_siku(ks, inertia) -> int:
    """Titik siku = jarak tegak lurus terjauh dari garis ujung ke ujung."""
    x = np.asarray(ks, dtype=float)
    y = np.asarray(inertia, dtype=float)
    x_n = (x - x.min()) / (x.max() - x.min())
    y_n = (y - y.min()) / (y.max() - y.min())
    x1, y1, x2, y2 = x_n[0], y_n[0], x_n[-1], y_n[-1]
    jarak = np.abs((y2 - y1) * x_n - (x2 - x1) * y_n + x2 * y1 - y2 * x1)
    jarak /= np.hypot(y2 - y1, x2 - x1)
    return int(x[int(np.argmax(jarak))])


def profil_relatif(df: pd.DataFrame, label) -> pd.DataFrame:
    """Simpangan rata-rata tiap klaster terhadap rata-rata seluruh data,
    dalam satuan simpangan baku. Baris = fitur, kolom = klaster."""
    d = df.copy()
    d['cluster'] = label
    rata = d.drop(columns='cluster').mean()
    sd = d.drop(columns='cluster').std().replace(0, np.nan)
    return ((d.groupby('cluster').mean() - rata) / sd).round(2).T


def nama_otomatis(profil: pd.DataFrame) -> dict[str, str]:
    """Dua fitur dengan simpangan terbesar secara mutlak, beserta arahnya.
    WAJIB diganti mahasiswa dengan nama yang bermakna."""
    nama = {}
    for c in profil.columns:
        kol = profil[c]
        kuat = kol.reindex(kol.abs().sort_values(ascending=False).index)[:2]
        nama[str(int(c))] = ', '.join(
            f'{n} {"tinggi" if v > 0 else "rendah"}' for n, v in kuat.items())
    return nama


def simpan_hasil(nomor: int, isi: dict) -> str:
    os.makedirs(FOLDER_HASIL, exist_ok=True)
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    with open(jalur, 'w', encoding='utf-8') as f:
        json.dump(isi, f, indent=2, ensure_ascii=False)
    return jalur


def baca_hasil(nomor: int) -> dict | None:
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    if not os.path.exists(jalur):
        return None
    with open(jalur, encoding='utf-8') as f:
        return json.load(f)

Overwriting alat.py


In [5]:
def ambil_berkas():
    """Cadangan: unggah berkas manual bila folder Drive tak bisa diakses."""
    from google.colab import files
    for nama in files.upload():
        if nama.startswith('varian') and nama.endswith('.json'):
            tujuan = 'data/hasil_varian/' + nama
        elif nama.endswith('.pkl') or nama == 'segmen.json':
            tujuan = 'model/' + nama
        elif nama.endswith('.png'):
            tujuan = 'gambar/' + nama
        else:
            tujuan = 'data/' + nama
        shutil.move(nama, tujuan)
        print('->', tujuan)


def perlu(*jalur):
    """Memastikan luaran pertemuan sebelumnya sudah ada di folder Drive."""
    kurang = [j for j in jalur if not os.path.exists(j)]
    if not kurang:
        print('Lengkap:', ', '.join(jalur))
        return
    raise FileNotFoundError(
        'Belum ada di folder Drive: ' + ', '.join(kurang) + '. '
        'Jalankan notebook pertemuan sebelumnya sampai selesai. Bila folder '
        'kelompok tidak dapat diakses dari akun Google Anda, jalankan '
        'ambil_berkas() di sel baru untuk mengunggah berkasnya manual.')


print('Folder kerja:', os.getcwd())
print('perlu() dan ambil_berkas() siap dipakai.')

Folder kerja: /content
perlu() dan ambil_berkas() siap dipakai.


## 4.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan di folder Drive kelompok. Bila ada yang belum dihasilkan, sel berhenti dengan pesan yang menyebut berkasnya. Aman dijalankan berulang kali.

In [ ]:
perlu('data/data_bersih.csv')

Lengkap: data/data_bersih.csv


## 4.2 Menjalankan varian 3

In [10]:
import sys
import os
import shutil

# Kembalikan ke /content terlebih dahulu untuk pengerjaan lokal yang aman
os.chdir('/content')
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Tulis langsung alat.py di /content untuk memastikan modul selalu tersedia lokal
with open('/content/alat.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"alat.py -- fungsi bersama ketiga varian. Jangan diubah sendiri-sendiri.\"\"\"
from __future__ import annotations

import json
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (calinski_harabasz_score,
                             davies_bouldin_score, silhouette_score)
from sklearn.preprocessing import StandardScaler

SEED = 42
RENTANG_K = range(2, 11)
BERKAS_BERSIH = "data/data_bersih.csv"
FOLDER_HASIL = "data/hasil_varian"

def muat_bersih(fitur: list[str] | None = None) -> pd.DataFrame:
    df = pd.read_csv(BERKAS_BERSIH)
    df = df.select_dtypes(include=\'number\')
    return df[fitur] if fitur else df

def skalakan(df: pd.DataFrame):
    sc = StandardScaler()
    return sc.fit_transform(df), sc

def jalankan_k(X_scaled, rentang=RENTANG_K) -> pd.DataFrame:
    baris = []
    for k in rentang:
        km = KMeans(n_clusters=k, init=\'k-means++\', n_init=20, random_state=SEED)
        lab = km.fit_predict(X_scaled)
        baris.append({
            \'k\': k,
            \'inertia\': float(km.inertia_),
            \'silhouette\': float(silhouette_score(X_scaled, lab)),
            \'davies_bouldin\': float(davies_bouldin_score(X_scaled, lab)),
            \'calinski_harabasz\': float(calinski_harabasz_score(X_scaled, lab)),
        })
    return pd.DataFrame(baris)

def simpan_hasil(nomor: int, isi: dict) -> str:
    os.makedirs(FOLDER_HASIL, exist_ok=True)
    jalur = f\'{FOLDER_HASIL}/varian{nomor}.json\'
    with open(jalur, \'w\', encoding=\'utf-8\') as f:
        json.dump(isi, f, indent=2, ensure_ascii=False)
    return jalur

def baca_hasil(nomor: int) -> dict | None:
    jalur = f\'{FOLDER_HASIL}/varian{nomor}.json\'
    if not os.path.exists(jalur):
        return None
    with open(jalur, encoding=\'utf-8\') as f:
        return json.load(f)
''')

# Buat struktur direktori lokal yang dibutuhkan alat.py
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/gambar', exist_ok=True)
os.makedirs('/content/model', exist_ok=True)

# Salin data_bersih.csv ke data/data_bersih.csv agar muat_bersih() di alat.py menemukannya
if os.path.exists('/content/data_bersih.csv'):
    shutil.copy('/content/data_bersih.csv', '/content/data/data_bersih.csv')

import matplotlib
import matplotlib.pyplot as plt
from alat import baca_hasil, jalankan_k, muat_bersih, simpan_hasil, skalakan

print('=' * 70)
print('VARIAN 3 - PENENTUAN k DENGAN DAVIES-BOULDIN INDEX')
print('=' * 70)

df = muat_bersih()
X, _ = skalakan(df)
print(f'Data bersih kelompok: {df.shape[0]} baris, {df.shape[1]} fitur')
print('Fitur dan penskalaan dikunci sama dengan Varian 1 dan Varian 2,')
print('sehingga yang dibandingkan benar-benar kriteria pemilihan k.\n')

tabel = jalankan_k(X)
print(tabel[['k', 'davies_bouldin', 'calinski_harabasz', 'silhouette', 'inertia']]
      .round(4).to_string(index=False))

k_dbi = int(tabel.loc[tabel['davies_bouldin'].idxmin(), 'k'])
dbi_terbaik = float(tabel['davies_bouldin'].min())
k_ch = int(tabel.loc[tabel['calinski_harabasz'].idxmax(), 'k'])
ch_terbaik = float(tabel['calinski_harabasz'].max())

print(f'\nDavies-Bouldin terendah  : k = {k_dbi}  ({dbi_terbaik:.4f})')
print(f'Calinski-Harabasz tertinggi: k = {k_ch}  ({ch_terbaik:.1f})')

if dbi_terbaik <= 0.8:
    tafsir = 'klaster rapat di dalam dan berjauhan antarklaster'
elif dbi_terbaik <= 1.5:
    tafsir = 'pemisahan sedang, masih dapat dipertanggungjawabkan'
else:
    tafsir = 'sebaran dalam klaster hampir sebesar jarak antarpusat'
print(f'Tafsiran nilai DBI       : {tafsir}')

print('\n' + '-' * 70)
print('APA YANG DIUKUR KEDUA INDEKS INI')
print('-' * 70)
print('Davies-Bouldin  : rata-rata, untuk tiap klaster, dari rasio')
print('                  (sebaran dalam klaster + sebaran tetangga terdekat)')
print('                  dibagi jarak antarpusatnya.')
print('                  MAKIN KECIL MAKIN BAIK. Nilai 0 berarti sempurna.')
print('Calinski-Harabasz: rasio keragaman antarklaster terhadap keragaman')
print('                  di dalam klaster, disesuaikan dengan jumlah klaster.')
print('                  MAKIN BESAR MAKIN BAIK.')
print('Keduanya memakai gagasan yang sama - rapat di dalam, renggang di luar -')
print('tetapi arah bacanya berlawanan. Karena itu keduanya dipakai bersama:')
print('kalau keduanya menunjuk k yang sama, pilihan itu lebih kuat.')

if k_dbi == k_ch:
    sepakat = True
    catatan = (f'Kedua indeks sepakat pada k={k_dbi}, sehingga pilihan ini '
               f'berdiri di atas dua perhitungan yang saling menguatkan.')
else:
    sepakat = False
    dbi_di_ch = float(tabel.loc[tabel['k'] == k_ch, 'davies_bouldin'].iloc[0])
    catatan = (f'Kedua indeks berbeda: DBI menunjuk k={k_dbi}, '
               f'Calinski-Harabasz menunjuk k={k_ch}. Selisihnya perlu '
               f'dijelaskan, bukan diabaikan. Pada k={k_ch} nilai DBI adalah '
               f'{dbi_di_ch:.4f}, selisih {dbi_di_ch - dbi_terbaik:+.4f} dari '
               f'nilai terendah. Calinski-Harabasz cenderung naik pada k kecil '
               f'karena pembaginya ikut mengecil, sehingga pada data yang '
               f'strukturnya lemah ia sering memuncak lebih awal daripada DBI.')
print(f'\n{catatan}')
print('-' * 70)

lain = {n: baca_hasil(n) for n in (1, 2)}
print('\nPerbandingan dengan varian lain:')
if not any(lain.values()):
    print('  Belum ada hasil Varian 1 atau Varian 2 di sesi ini.')
    print('  Perbandingan ketiga kriteria disusun kelompok di Bagian 5.')
else:
    nama_varian = {1: 'Varian 1 (Elbow)', 2: 'Varian 2 (Silhouette)'}
    for n, isi in lain.items():
        if isi:
            print(f'  {nama_varian[n]:<26} k = {isi["k_terpilih"]}')
    print(f'  {"Varian 3 (Davies-Bouldin)":<26} k = {k_dbi}')

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(tabel['k'], tabel['davies_bouldin'], marker='o', color='tab:red')
plt.axvline(k_dbi, ls='--', lw=1, color='grey')
plt.xlabel('k'); plt.ylabel('Davies-Bouldin Index')
plt.title(f'Varian 3 - Davies-Bouldin (terendah di k={k_dbi})')
plt.subplot(1, 2, 2)
plt.plot(tabel['k'], tabel['calinski_harabasz'], marker='o', color='tab:purple')
plt.axvline(k_ch, ls='--', lw=1, color='grey')
plt.xlabel('k'); plt.ylabel('Calinski-Harabasz')
plt.title(f'Pemeriksa silang (tertinggi di k={k_ch})')
plt.tight_layout(); plt.savefig('gambar/varian3_davies_bouldin.png', dpi=110)
plt.close()

jalur = simpan_hasil(3, {
    'varian': 3,
    'nama': 'Penentuan k dengan Davies-Bouldin Index',
    'kriteria': 'Davies-Bouldin Index terendah',
    'pemeriksa_silang': 'Calinski-Harabasz Index tertinggi',
    'fitur_dipakai': df.columns.tolist(),
    'k_terpilih': k_dbi,
    'k_menurut_calinski_harabasz': k_ch,
    'kedua_indeks_sepakat': sepakat,
    'tafsiran': tafsir,
    'catatan': catatan,
    'metrik_utama': {'davies_bouldin': round(dbi_terbaik, 4)},
    'metrik_pendukung': {
        'calinski_harabasz': round(ch_terbaik, 1),
        'silhouette': round(float(tabel.loc[tabel['k'] == k_dbi, 'silhouette'].iloc[0]), 4),
        'inertia': round(float(tabel.loc[tabel['k'] == k_dbi, 'inertia'].iloc[0]), 2),
    },
    'tabel_k': tabel[['k', 'inertia', 'silhouette', 'davies_bouldin',
                      'calinski_harabasz']].round(4).to_dict(orient='records'),
})

# Salin hasil kembali ke folder Drive agar tersinkronisasi
if os.path.exists(FOLDER):
    try:
        shutil.copy(jalur, os.path.join(FOLDER, jalur))
        shutil.copy('gambar/varian3_davies_bouldin.png', os.path.join(FOLDER, 'gambar/varian3_davies_bouldin.png'))
        print('Tersinkronisasi ke Drive Kelompok!')
    except Exception as e:
        print('Gagal sinkronisasi otomatis ke Drive:', e)

print(f'\nTersimpan: {jalur}  dan  gambar/varian3_davies_bouldin.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 3 - PENENTUAN k DENGAN DAVIES-BOULDIN INDEX
Data bersih kelompok: 9659 baris, 3 fitur
Fitur dan penskalaan dikunci sama dengan Varian 1 dan Varian 2,
sehingga yang dibandingkan benar-benar kriteria pemilihan k.

 k  davies_bouldin  calinski_harabasz  silhouette    inertia
 2          0.6098         12698.2676      0.5915 12517.4475
 3          0.6151         15618.5760      0.5654  6842.2682
 4          0.7347         14308.3293      0.4948  5320.9015
 5          0.7610         14331.2349      0.5203  4176.5973
 6          0.8139         13895.5284      0.5251  3534.8548
 7          0.8089         14118.2810      0.5137  2963.9910
 8          0.7498         15017.0834      0.5403  2436.6809
 9          0.7314         15385.0799      0.5354  2106.7766
10          0.7576         15648.7373      0.5397  1857.9615

Davies-Bouldin terendah  : k = 2  (0.6098)
Calinski-Harabasz tertinggi: k = 10  (15648.7)
Tafsiran nilai DBI       : klaster rapat di dalam dan berjauhan antarklaster

--

## 4.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. **Apa yang sebenarnya diukur Davies-Bouldin, dengan kalimat Anda sendiri?**
     DBI itu buat ukur seberapa bagus pengelompokan data kita. dia melihat dua hal yaitu seberapa padat data di dalam satu kelompok dan seberapa jauh jarak antar kelompok itu. Target kita adalah nyari skor sekecil mungkin "makin dekat ke nol, makin bagus nilainya".

2. **Apakah Davies-Bouldin dan Calinski-Harabasz menunjuk k yang sama?**
   tida sama. hasil hitungan DBI disarankan pake k=2 (skornya paling kecil, 0.6098). Sedangkan CH malah milih k=10 (skornya paling gede, 15648.7). Beda gini wajar banget karena data aplikasi kita gak punya batas yang kaku, dan CH emang punya kecenderungan buat naik terus pas nilai k-nya makin gede.

3. **Apa arti nilai DBI Anda dalam konteks kasus kelompok?**
   Skor DBI terbaik kita kan dapat 0.6098. Menurut aturan kelas, angka di bawah 0.8 itu artinya kelompok data kita udah terpisah dengan jelas dan padat di dalem. Jadi, ngebagi data Google Play Store kelompok kita (pake fitur Rating, Reviews, dan Installs) jadi **2 kelompok** itu udah pas banget dan keliatan bedanya.

4. **Dugaan Anda:** apakah Elbow dan Silhouette akan menunjuk k yang sama?
     silhouette lebih milih k=2, soalnya di tabel emang nilai tertingginya ada di situ (0.5915). Tapi buat metode Elbow (Inersia), tebakan saya sikunya bakal keliatan di sekitar k=3 atau k=4. Soalnya dari k=2 ke k=3 penurunan nilai inersianya drastis (hampir separuh), setelah itu grafiknya mulai landai.

## 4.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 4, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian3.json')
files.download('gambar/varian3_davies_bouldin.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_3.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Kirim `varian3.json` ke kelompok untuk Bagian 5.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_3.ipynb,
     lalu commit ke folder notebook/.
  3. Kirim `varian3.json` ke kelompok untuk Bagian 5.


## Checklist Bagian 4

- [x] `varian3.json` ter-commit di `data/hasil_varian/`
- [x] `varian3_davies_bouldin.png` ter-commit di `gambar/`
- [x] Notebook ini diunduh, diberi nama `03_varian_3.ipynb`, dan ter-commit di `notebook/`
- [x] Commit dilakukan dengan akun GitHub Anggota 3 sendiri
- [x] Keempat jawaban di 4.3 sudah ditulis
- [x] Dugaan tentang hasil anggota lain ditulis sebelum melihat hasilnya
- [x] `varian3.json` sudah dikirim ke kelompok